In [ ]:
# imports

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from copy import copy
from mpl_toolkits import axes_grid1

In [ ]:
# importing latex for text formatting

from matplotlib import rc
rc('text', usetex=True)

In [ ]:
# get data to plot

example = 2

if example == 1:
    # BPix example
    year = '2025'
    run = 392250
    lumi = 633
    layer = 'BPix1'
    
elif example == 2:
    # FPix example
    year = '2025'
    run = 396871
    lumi = 743
    layer = 'PXRing2'

fname = f'data_year{year}_run{run}_ls{lumi}_layer{layer}.npz'
data = np.load(fname)
orig = data['orig']
loss = data['loss']
binary = data['binary']

In [ ]:
# define plotting functions
# (copied and modified from elsewhere in the framework)

def add_colorbar(fig, ax, cobject, width=0.05, pad=0.03, **kwargs):
    cax = ax.inset_axes([1 + pad, 0, width, 1])
    return fig.colorbar(cobject, cax=cax, **kwargs)


def plot_hist_2d(hist, fig=None, ax=None, figsize=None, title=None, titlesize=None,
                xaxtitle=None, xaxtitlesize=None, yaxtitle=None, yaxtitlesize=None,
                ticklabelsize=None, colorticklabelsize=None, extent=None, aspect=None, box_aspect=None,
                caxrange=None, caxsoftmax=False,
                docolorbar=True, caxtitle=None, caxtitlesize=None, caxtitleoffset=None,
                origin='lower'):
    
    if fig is None or ax is None: fig,ax = plt.subplots(figsize=figsize)
    
    # settings
    histmin = np.amin(hist)
    histmax = np.amax(hist)
    histsoftmax = np.quantile(hist.astype(float), 0.99)
    hasnegative = histmin < -1e-6
    if aspect is None: aspect = 'equal'
    if extent is not None: aspect = 'auto'
    # if a box aspect is given (height/width of the plotted area, e.g. 1 for a square),
    # let the pixels stretch to fill a box of that shape, regardless of the input dimensions
    if box_aspect is not None:
        aspect = 'auto'
        ax.set_box_aspect(box_aspect)        
    # make color object
    if not hasnegative:
        vmin = 1e-6
        vmax = max(vmin*2, histmax)
        if caxsoftmax: vmax = max(vmin*2, histsoftmax)
        norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax, clip=False)
    else: 
        extremum = max(abs(histmax),abs(histmin))
        norm = mpl.colors.Normalize(vmin=-extremum, vmax=extremum, clip=False)
    if caxrange is not None:
        norm = mpl.colors.Normalize(vmin=caxrange[0], vmax=caxrange[1], clip=False)
    cmap = copy(mpl.colormaps['jet'])
    cmap.set_under('w')
    cobject = mpl.cm.ScalarMappable(norm=norm, cmap=cmap) # needed for colorbar
    cobject.set_array([]) # ad-hoc bug fix
    
    # make the plot
    ax.imshow(hist, aspect=aspect, interpolation='none', norm=norm, cmap=cmap,
              extent=extent, origin=origin)
    
    # add the colorbar
    if docolorbar:
        cbar = add_colorbar(fig, ax, cobject)
        if caxtitleoffset is not None: cbar.ax.get_yaxis().labelpad = caxtitleoffset
        if caxtitle is not None: cbar.ax.set_ylabel(caxtitle, fontsize=caxtitlesize, rotation=270)
    
    # add titles
    if ticklabelsize is not None: ax.tick_params(labelsize=ticklabelsize)
    if docolorbar and colorticklabelsize is not None: cbar.ax.tick_params(labelsize=colorticklabelsize)
    if title is not None: ax.set_title(title, fontsize=titlesize)
    if xaxtitle is not None: ax.set_xlabel(xaxtitle, fontsize=xaxtitlesize)
    if yaxtitle is not None: ax.set_ylabel(yaxtitle, fontsize=yaxtitlesize)
    return (fig,ax)


def plot_cluster_occupancy(hist, fig=None, ax=None, fast=False,
                           do_xaxtitle=True, xaxtitlesize=None,
                           do_yaxtitle=True, yaxtitlesize=None,
                           **kwargs):
    
    # default arguments
    default_kwargs = {
        'extent': None,
        'aspect': None,
        'box_aspect': 1,
        'origin': 'lower'
    }
    default_kwargs.update(kwargs)
    kwargs = default_kwargs
   
    # make base plot
    fig, ax = plot_hist_2d(hist,
                # fixed arguments
                fig=fig, ax=ax,
                # modifiable arguments
                **kwargs)
    
    # for faster plotting, don't bother with further aesthetics
    if fast: return fig, ax
    
    # automatically determine x-axis labels from number of bins
    nxbins = hist.shape[1]
    axvlines = np.arange(8-0.5, nxbins-0.5, 8)
    xaxticks = np.arange(4-0.5, nxbins-0.5, 8)
    if nxbins==72:
        # BPix, raw
        xaxtitle = 'Module'
        xaxticklabels = np.arange(-4, 5)
    elif nxbins==64:
        # BPix, preprocessed
        xaxtitle = 'Module'
        xaxticklabels = np.concatenate((np.arange(-4, 0), np.arange(1, 5)))
    elif nxbins==56:
        # FPix, raw
        xaxtitle = 'Disk'
        xaxticklabels = np.arange(-3, 4)
    elif nxbins==48:
        # FPix, preprocessed
        xaxtitle = 'Disk'
        xaxticklabels = np.concatenate((np.arange(-3, 0), np.arange(1, 4)))
    else: raise Exception('Number of x-bins not recognized.')
        
    # x-axis layout
    if do_xaxtitle: ax.set_xlabel(xaxtitle, fontsize=xaxtitlesize)
    for x in axvlines: ax.axvline(x, color='black', linestyle='dotted')
    ax.set_xticks(ticks=xaxticks, labels=xaxticklabels)
        
    # automatically determine y-axis labels from number of bins
    nybins = hist.shape[0]
    if nxbins==72 or nxbins==64:
        # BPix
        axhlines = np.arange(2-0.5, nybins-0.5, 2)
        yaxticks = np.arange(1-0.5, nybins-0.5, 2)
    elif nxbins==56 or nxbins==48:
        # FPix
        axhlines = np.arange(4-0.5, nybins-0.5, 4)
        yaxticks = np.arange(2-0.5, nybins-0.5, 4)
    if nxbins==72:
        # BPix, raw
        yaxtitle = 'Ladder'
        nladders = int((nybins-2)/4)
        yaxticklabels = np.arange(-nladders, nladders+1)
    elif nxbins==64:
        # BPix, preprocessed
        yaxtitle = 'Ladder'
        nladders = int(nybins/4)
        yaxticklabels = np.concatenate((np.arange(-nladders, 0), np.arange(1, nladders+1)))
    elif nxbins==56:
        # FPix, raw
        yaxtitle = 'Panel'
        nladders = int((nybins-4)/8)
        yaxticklabels = np.arange(-nladders, nladders+1)
    elif nxbins==48:
        # FPix, preprocessed
        yaxtitle = 'Panel'
        nladders = int(nybins/8)
        yaxticklabels = np.concatenate((np.arange(-nladders, 0), np.arange(1, nladders+1)))
    else: raise Exception('Number of y-bins not recognized.')
        
    # subsample y-axis labels (else they are typically overlapping)
    stepdict = {6: 2, 14: 2, 22: 4, 32: 4, 17: 2}
    step = stepdict.get(nladders, 1)
    ids = np.sort(np.concatenate((np.arange(0, int((len(yaxticks)+1)/2), step),
                                  np.arange(len(yaxticks)-1, int(len(yaxticks)/2), -step))))
    yaxticks = yaxticks[ids]
    yaxticklabels = yaxticklabels[ids]
        
    # y-axis layout
    if do_yaxtitle: ax.set_ylabel(yaxtitle, fontsize=yaxtitlesize)
    for y in axhlines: ax.axhline(y, color='black', linestyle='dotted')
    ax.set_yticks(ticks=yaxticks, labels=yaxticklabels)
        
    return fig, ax

In [ ]:
# make plot

# initialize figure
nrows = 1
fig, axs = plt.subplots(ncols=3, nrows=nrows, figsize=(24, 8), squeeze=False)
            
# plot raw data
fig, axs[0, 0] = plot_cluster_occupancy(orig, fig=fig, ax=axs[0, 0],
                   title='Input',
                   titlesize=25,
                   xaxtitlesize=25, yaxtitlesize=25,
                   ticklabelsize=20, colorticklabelsize=20,
                   caxsoftmax=True,
                   docolorbar=True, caxtitle='Number of clusters',
                   caxtitlesize=25, caxtitleoffset=30)
fig, axs[0, 1] = plot_cluster_occupancy(loss, fig=fig, ax=axs[0, 1],
                   title='Loss', titlesize=25,
                   xaxtitlesize=25, do_yaxtitle=False, yaxtitlesize=15,
                   ticklabelsize=20, colorticklabelsize=20,
                   docolorbar=True, caxtitle='Loss',
                   caxrange=(0, 0.1),
                   caxtitlesize=25, caxtitleoffset=25)
fig, axs[0, 2] = plot_cluster_occupancy(binary, fig=fig, ax=axs[0, 2],
                   title=f'Cleaned loss', titlesize=25,
                   xaxtitlesize=25, do_yaxtitle=False, yaxtitlesize=15,
                   ticklabelsize=20, colorticklabelsize=20,
                   docolorbar=True, caxtitle='Loss',
                   caxrange=(0, 1),
                   caxtitlesize=25, caxtitleoffset=25)

# spacing
fig.subplots_adjust(wspace=0.45)
    
# title
title = r'$\textbf{CMS}$'
conditions = f'{year} (13.6 TeV)'
if str(layer)=='BPix1' or str(layer)=='PXRing2':
    axs[0, 0].text(0.01, 1.15, title, fontsize=25, transform=axs[0, 0].transAxes)
    axs[0, 1].text(0.5, 1.15, layer, fontsize=25, ha='center', transform=axs[0,1].transAxes)
    axs[0, 2].text(0.99, 1.15, conditions, fontsize=25, ha='right', transform=axs[0,2].transAxes)
            
fig.savefig(f'pixel_example{example}.png', bbox_inches='tight')
fig.savefig(f'pixel_example{example}.pdf', bbox_inches='tight')
plt.show()
plt.close()